# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and performing basic analysis on the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (FAIR² mental health survey Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We list all record sets and key fields referenced via their `@id` to facilitate precise manipulations.

__Note__: in Croissant, recordSets can be found via `metadata.record_sets` (a list of RecordSet objects).

In [ ]:
# List all record sets and their field @ids
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for i, record_set in enumerate(record_sets):
    print(f"Record Set {i+1}:")
    print(f"  @id        : {record_set.id}")
    print(f"  name       : {record_set.name}")
    print(f"  description: {record_set.description}")

    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    @id: {field.id} | name: {field.name} | type: {field.data_type}")
    print("-")

## 3. Data Extraction
Load data from a specific record set (`@id`) into a DataFrame for analysis.

Below, we loop over all record sets and show an example extraction from the main survey record set (usually containing GAD-7, PHQ-9, PSQ scores and demographics).

In [ ]:
# Discover all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print(f"Available record set @ids: {record_set_ids}\n")

# We expect the main table to hold the survey. Let's use the first (or most relevant) record set.
# Replace below with the actual @id you want, e.g. 'cr:KilifiMentalHealthSurvey_RS'
main_record_set_id = record_set_ids[0]  # or specify if you know

print(f"Loading data from main record set: {main_record_set_id}")

df_dict = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df_dict[rsid] = pd.DataFrame(records)

# List the columns from the main survey DataFrame
print("Fields from the main record set:")
print(df_dict[main_record_set_id].columns.tolist())
df_dict[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We demonstrate filtering, normalization and grouping on a numeric field. Use field `@id` from the above printout or Croissant metadata (e.g., 'cr:GAD7_score' or numeric symptom score fields).

__Example__: Filter for participants with PHQ-9 scores greater than a threshold and group by `gender`.

In [ ]:
# Example EDA: use the correct field IDs for PHQ-9 and gender (adjust for actual @id in this dataset)
# Replace these with actual IDs if available in your overview
numeric_field = None
group_field = None
for field in record_sets[0].fields:
    # Try to find the PHQ-9 score, GAD-7, or another relevant numeric field.
    if ('phq' in field.name.lower() or 'gad' in field.name.lower() or 'psq' in field.name.lower()) and 'score' in field.name.lower():
        numeric_field = field.id
    if 'gender' in field.name.lower() or 'sex' in field.name.lower():
        group_field = field.id
print(f"Detected numeric_field = {numeric_field}")
print(f"Detected group_field = {group_field}")

# Fallbacks if auto-detection failed (edit as needed based on printed columns/metadata)
if not numeric_field:
    numeric_field = df_dict[main_record_set_id].columns[0]  # Or set manually
if not group_field:
    group_field = df_dict[main_record_set_id].columns[1]    # Or set manually

# Proceed if numeric_field exists in the data
if numeric_field in df_dict[main_record_set_id].columns:
    threshold = 10  # Example threshold
    filtered_df = df_dict[main_record_set_id][df_dict[main_record_set_id][numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field (e.g., gender) if it exists
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())
else:
    print(f"No numeric field '{numeric_field}' found in {main_record_set_id} for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field (e.g., PHQ-9 score) and group comparison (e.g., by gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check if numeric_field is in DataFrame
df = df_dict[main_record_set_id]
if numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field also available, show comparison
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
We have demonstrated how to load and explore the FAIR² Mental Health Survey Dataset using Croissant tools, showing how to reference, extract, and analyze data by record set and field `@id`. For further analysis, consult the Croissant schema and data documentation for specific field definitions and analytic guidance.